# explororation 16 : 다음에 볼 영화 예측하기

**루브릭 1.  Movielens 데이터셋을 session based recommendation 관점으로 전처리하는 과정이 체계적으로 진행되었다.**  
데이터셋의 면밀한 분석을 토대로 세션단위 정의 과정(길이분석, 시간분석)을 합리적으로 수행한 과정이 기술되었다.

-> 

**루브릭 2. RNN 기반의 예측 모델이 정상적으로 구성되어 안정적으로 훈련이 진행되었다.**  
적절한 epoch만큼의 학습이 진행되는 과정에서 train loss가 안정적으로 감소하고, validation 단계에서의 Recall, MRR이 개선되는 것이 확인된다.

-> 

**루브릭 3. 세션정의, 모델구조, 하이퍼파라미터 등을 변경해서 실험하여 Recall, MRR 등의 변화추이를 관찰하였다.**  
3가지 이상의 변화를 시도하고 그 실험결과를 체계적으로 분석하였다.

-> 

## 16-1. 들어가며
### Session-Based Recommendation
 - 고객의 선호가 고정되어 있는가? 아니다, 변덕스럽다.
 - '바로 지금 시점에 이 고객이 좋아할 만한상품'?
 - SBR : 유저가 바로 다음에 클릭 또는 구매할 아이템을 예측하는 추천
 - Session : 유저가 서비스를 이용하면서 발생하는 중요한 정보를 담은 데이터 (서버에 저장)
 - 유저의 행동 데이터들은 쿠키의 형태로 저장 (유저 측 브라우저)
 - 쿠키는 세션과 상호작용하며 정보를 주고받는다.
 - [Def] Session : 브라우저가 종료되기 전까지 유저의 행동을 담은 시퀀스 데이터
 - 유저 : 9194111
 - 상품 : 214853420 → 214850942 → 214850942 → 214829878
 - 학회 RecSys, 추천 엔진 솔루션 회사 2015 YOOCHOOSE, 데이터 E-Commerce
 - (코드) : description
 - yoochoose-clicks.dat : Session ID / Timestamp / Item ID / Category
 - yoochoose-buys.dat : Session ID / Timestamp / Item ID / Price / Quantity
 - yoochoose-test.dat : identical to the file youchoose.dat except for the buying events
 - Sequential Recommendation : 유저에 대한 정보를 알 수 있는 경우 (cf. Context-Aware)

## 16-2. Datapreprocess

### 16-2-1. Data Load

In [1]:
# import modules
import datetime as dt
from pathlib import Path
import os

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [2]:
# 경로지정 : yoochoose-clicks.data
data_path = Path(os.getenv('HOME')+'/aiffel/yoochoose/data') 
train_path = data_path / 'yoochoose-clicks.dat'
train_path

PosixPath('/aiffel/aiffel/yoochoose/data/yoochoose-clicks.dat')

In [3]:
# load_data 함수
def load_data(data_path: Path, nrows=None):
    data = pd.read_csv(data_path, sep=',', header=None, usecols=[0, 1, 2],
                       parse_dates=[1], dtype={0: np.int32, 2: np.int32}, nrows=nrows)
    data.columns = ['SessionId', 'Time', 'ItemId']
    return data

In [4]:
# data (pd.DataFrame, 33003944 rows, 3 columns)
# 시간이 좀 걸릴 수 있습니다. 메모리도 10GB 가까이 소요될 수 있으니 메모리 상태에 주의해 주세요.  
import time
start = time.time()

data = load_data(train_path, None)
data.sort_values(['SessionId', 'Time'], inplace=True)  # data를 id와 시간 순서로 정렬해줍니다.
data

print("time :", time.time() - start)

time : 154.523131608963


In [5]:
# # save the data in the aiffel cloud
# data.to_csv('../data/exploration_16/data.csv')

In [6]:
# # run this cell to reload the original 'data'(i.e. 'yoochoose-clicks.dat' with the first three columns)
# # not recommended since the datatype for timestamp is converted into string.
# data = pd.read_csv('../data/exploration_16/data.csv')

In [7]:
data

,SessionId,Time,ItemId
0,1,2014-04-07 10:51:09.277000+00:00,214536502
1,1,2014-04-07 10:54:09.868000+00:00,214536500
2,1,2014-04-07 10:54:46.998000+00:00,214536506
3,1,2014-04-07 10:57:00.306000+00:00,214577561
4,2,2014-04-07 13:56:37.614000+00:00,214662742
...,...,...,...
32230487,11562158,2014-09-26 04:50:29.172000+00:00,214849132
32230488,11562158,2014-09-26 04:52:21.900000+00:00,214854774
32230489,11562158,2014-09-26 05:16:32.904000+00:00,214849132
32230490,11562159,2014-09-26 19:16:28.897000+00:00,214849132


In [8]:
data_duplicated = data

In [9]:
# 세션 수(유저 수), 아이템 수  = (9249729, 52739)
data['SessionId'].nunique(), data['ItemId'].nunique()

(9249729, 52739)

In [10]:
# save the data just after loading as data_after_2_1
# raw data
data_after_2_1 = data

### 16-2-2. Session Length

In [11]:
# 여기에서는 session length를 관찰하기만 하지 data를 따로 업데이트 하지는 않는다.
# 만약 필요하다면, quantile 99.9% 이하의 것들만 모아 새로운 data를 만들 수도 있겠다.

In [12]:
# 세션 길이의 중앙값, 평균 (2, 3.59)
# 세션 길이의 최솟값, 최댓값 (1, 200)
# data_after_2_1 (1, 200, 2.0, 3.568098481587947, 33003944, 9249729, 52739)
session_length = data_after_2_1.groupby('SessionId').size()
session_length.min(), session_length.max(), session_length.median(), session_length.mean(), len(data), data['SessionId'].nunique(), data['ItemId'].nunique()

(1, 200, 2.0, 3.568098481587947, 33003944, 9249729, 52739)

In [13]:
# # 세션 길이의 99.9% quantile (41)
session_length.quantile(0.999)

41.0

In [14]:
# # 세션 길이가 200인 세션에 대한 관찰 / 이상하다, 제거하자
# long_session = session_length[session_length==200].index[0]
# data[data['SessionId']==long_session]

In [15]:
# print(type(session_length))
# print(type(session_length[session_length==200]))
# print(session_length[session_length==200].index)
# print(session_length[session_length==200].index[0])

In [16]:
# # length_percent_cumsum : cumulative distribution of length_count where length_count is the counts of the session_length
# length_count = session_length.groupby(session_length).size()
# length_percent_cumsum = length_count.cumsum() / length_count.sum()
# length_percent_cumsum_999 = length_percent_cumsum[length_percent_cumsum < 0.999]

# length_percent_cumsum_999

In [17]:
# session_length.groupby(session_length).size()

In [18]:
# length_count.cumsum()

In [19]:
# # plt.bar / cumsum percentage until 0.999
# import matplotlib.pyplot as plt

# plt.figure(figsize=(12, 10))
# plt.bar(x=length_percent_cumsum_999.index,
#         height=length_percent_cumsum_999, color='red')
# plt.xticks(length_percent_cumsum_999.index)
# plt.yticks(np.arange(0, 1.01, 0.05))
# plt.title('Cumsum Percentage Until 0.999', size=20)
# plt.show()

### 16-2-3. Session Time

In [ ]:
# (Q2) 시간 관련 데이터의 다양한 측면 - 접속시간대, 접속시간, 접속요일, 계절, 마지막 접속시간 등

In [20]:
# inded, 6개월치의 데이터가 있다.
oldest, latest = data['Time'].min(), data['Time'].max()
print(oldest) 
print(latest)

2014-04-01 03:00:00.124000+00:00
2014-09-30 02:59:59.430000+00:00


In [21]:
# data['Time']의 각 객체들은 timestamp 데이터타입
# type(latest)

In [22]:
month_ago = latest - dt.timedelta(30)     # 최종 날짜로부터 30일 이전 날짜를 구한다.  
data = data[data['Time'] > month_ago]   # 방금 구한 날짜 이후의 데이터만 모은다. 
data

,SessionId,Time,ItemId
26837834,9194111,2014-08-31 17:40:46.805000+00:00,214853420
26837835,9194111,2014-08-31 17:42:26.089000+00:00,214850942
26837836,9194111,2014-08-31 17:44:06.583000+00:00,214829878
26837837,9194111,2014-08-31 17:48:49.873000+00:00,214853420
26838214,9194112,2014-09-01 13:26:36.292000+00:00,214853422
...,...,...,...
32230487,11562158,2014-09-26 04:50:29.172000+00:00,214849132
32230488,11562158,2014-09-26 04:52:21.900000+00:00,214854774
32230489,11562158,2014-09-26 05:16:32.904000+00:00,214849132
32230490,11562159,2014-09-26 19:16:28.897000+00:00,214849132


In [23]:
# save the data just after time-slicing as data_after_2_2
# raw data -> crop the last one month
data_after_2_3 = data

In [24]:
# data_after_2_1 (1, 200, 2.0, 3.568098481587947, 33003944, 9249729, 52739)
# data_after_2_3 (1, 200, 2.0, 3.441910225059669, 5641401, 1639032, 32461)
session_length = data_after_2_3.groupby('SessionId').size()
session_length.min(), session_length.max(), session_length.median(), session_length.mean(), len(data), data['SessionId'].nunique(), data['ItemId'].nunique()

(1, 200, 2.0, 3.441910225059669, 5641401, 1639032, 32461)

### 16-2-4. Data Cleansing

In [25]:
# short_session을 제거한 다음 unpopular item을 제거하면 다시 길이가 1인 session이 생길 수 있습니다.
# 이를 위해 반복문을 통해 지속적으로 제거 합니다.
def cleanse_recursive(data: pd.DataFrame, shortest, least_click) -> pd.DataFrame:
    while True:
        before_len = len(data)
        data = cleanse_short_session(data, shortest)
        data = cleanse_unpopular_item(data, least_click)
        after_len = len(data)
        if before_len == after_len:
            break
    return data


def cleanse_short_session(data: pd.DataFrame, shortest):
    session_len = data.groupby('SessionId').size()
    session_use = session_len[session_len >= shortest].index
    data = data[data['SessionId'].isin(session_use)]
    return data


def cleanse_unpopular_item(data: pd.DataFrame, least_click):
    item_popular = data.groupby('ItemId').size()
    item_use = item_popular[item_popular >= least_click].index
    data = data[data['ItemId'].isin(item_use)]
    return data

In [26]:
data = cleanse_recursive(data, shortest=2, least_click=5)
data

,SessionId,Time,ItemId
26837834,9194111,2014-08-31 17:40:46.805000+00:00,214853420
26837835,9194111,2014-08-31 17:42:26.089000+00:00,214850942
26837836,9194111,2014-08-31 17:44:06.583000+00:00,214829878
26837837,9194111,2014-08-31 17:48:49.873000+00:00,214853420
26838202,9194123,2014-08-31 19:26:57.386000+00:00,214601207
...,...,...,...
32230485,11562157,2014-09-25 12:31:10.391000+00:00,214580372
32230486,11562157,2014-09-25 12:31:29.679000+00:00,214516012
32230487,11562158,2014-09-26 04:50:29.172000+00:00,214849132
32230488,11562158,2014-09-26 04:52:21.900000+00:00,214854774


In [27]:
# save the data just after cleansing as data_after_2_4
# raw data -> crop the last one month -> cleansing
data_after_2_4 = data

In [28]:
# data_after_2_1 (1, 200, 2.0, 3.568098481587947, 33003944, 9249729, 52739)
# data_after_2_3 (1, 200, 2.0, 3.441910225059669, 5641401, 1639032, 32461)
# data_after_2_4 (2, 191, 3.0, 4.1337091840588105, 5254242, 1271072, 20158)
session_length = data_after_2_4.groupby('SessionId').size()
session_length.min(), session_length.max(), session_length.median(), session_length.mean(), len(data), data['SessionId'].nunique(), data['ItemId'].nunique()

(2, 191, 3.0, 4.1337091840588105, 5254242, 1271072, 20158)

In [29]:
data

,SessionId,Time,ItemId
26837834,9194111,2014-08-31 17:40:46.805000+00:00,214853420
26837835,9194111,2014-08-31 17:42:26.089000+00:00,214850942
26837836,9194111,2014-08-31 17:44:06.583000+00:00,214829878
26837837,9194111,2014-08-31 17:48:49.873000+00:00,214853420
26838202,9194123,2014-08-31 19:26:57.386000+00:00,214601207
...,...,...,...
32230485,11562157,2014-09-25 12:31:10.391000+00:00,214580372
32230486,11562157,2014-09-25 12:31:29.679000+00:00,214516012
32230487,11562158,2014-09-26 04:50:29.172000+00:00,214849132
32230488,11562158,2014-09-26 04:52:21.900000+00:00,214854774


In [30]:
# implement the previous codes manually
shortest=2
least_click=5
data = data_after_2_3
session_length = data.groupby('SessionId').size()
session_length.min(), session_length.max(), session_length.median(), session_length.mean(), len(data), data['SessionId'].nunique(), data['ItemId'].nunique()

(1, 200, 2.0, 3.441910225059669, 5641401, 1639032, 32461)

In [31]:
# cleanse_short_session (1)
before_len = len(data) # 5641401
session_len = data.groupby('SessionId').size() # sessionId의 도수분포 # of length 1639032 # sessionID는 1639032개 있다.
session_use = session_len[session_len >= shortest].index # of length 1277626 # sessionID의 도수가 2 이상인 것들의 index(sessionID)
data = data[data['SessionId'].isin(session_use)] # of length 5279995
# 5279995 = \sum_{세션=1}^{1277626}세션당 클릭수"
data_ss1 = data
session_length = data.groupby('SessionId').size()
session_length.min(), session_length.max(), session_length.median(), session_length.mean(), len(data), data['SessionId'].nunique(), data['ItemId'].nunique()

(2, 200, 3.0, 4.13266088824116, 5279995, 1277626, 31399)

In [32]:
# cleanse_unpopular_item (1)
item_popular = data.groupby('ItemId').size() # ItemId의 도수분포 # of length 31399 # item은 31399개 있다.
item_use = item_popular[item_popular >= least_click].index # of length 20280 # itemID의 도수가 5 이상인 것들의 index(ItemID)
data = data[data['ItemId'].isin(item_use)] # of length 5258344
# 5258344 = \sum_{item=1}^{20280}아이템별 클릭수
data_ui1 = data
session_length = data.groupby('SessionId').size()
session_length.min(), session_length.max(), session_length.median(), session_length.mean(), len(data), data['SessionId'].nunique(), data['ItemId'].nunique()

(1, 191, 3.0, 4.124880470227575, 5258344, 1274787, 20280)

In [33]:
after_len = len(data)
before_len - after_len

383057

In [34]:
# cleanse_short_session (2)
before_len = len(data) # 5258344
session_len = data.groupby('SessionId').size() # sessionId의 도수분포 # of length 1274787 # sessionID는 1274787개 있다.
session_use = session_len[session_len >= shortest].index # of length 1271234 # sessionID의 도수가 2 이상인 것들의 index(sessionID)
data = data[data['SessionId'].isin(session_use)] # of length 5254791
# 5254791 = \sum_{세션=1}^{1271234}세션당 클릭수"
data_ss2 = data
session_length = data.groupby('SessionId').size()
session_length.min(), session_length.max(), session_length.median(), session_length.mean(), len(data), data['SessionId'].nunique(), data['ItemId'].nunique()

(2, 191, 3.0, 4.133614267711531, 5254791, 1271234, 20280)

In [35]:
# cleanse_unpopular_item (2)
item_popular = data.groupby('ItemId').size() # ItemId의 도수분포 # of length 20280 # item은 20280개 있다.
item_use = item_popular[item_popular >= least_click].index # of length 20163 # itemID의 도수가 5 이상인 것들의 index(ItemID)
data = data[data['ItemId'].isin(item_use)] # of length 5254348
# 5254348 = \sum_{item=1}^{20163}아이템별 클릭수
data_ui2 = data
session_length = data.groupby('SessionId').size()
session_length.min(), session_length.max(), session_length.median(), session_length.mean(), len(data), data['SessionId'].nunique(), data['ItemId'].nunique()

(1, 191, 3.0, 4.133506403599862, 5254348, 1271160, 20163)

In [36]:
after_len = len(data)
before_len - after_len

3996

In [37]:
# cleanse_short_session (3)
before_len = len(data) # 5254348
session_len = data.groupby('SessionId').size() # sessionId의 도수분포 # of length 1271160 # sessionID는 1271160 개 있다.
session_use = session_len[session_len >= shortest].index # of length 1271079 # sessionID의 도수가 2 이상인 것들의 index(sessionID)
data = data[data['SessionId'].isin(session_use)] # of length 5254267  
# 5254267 = \sum_{세션=1}^{1271079}세션당 클릭수"
data_ss3 = data
session_length = data.groupby('SessionId').size()
session_length.min(), session_length.max(), session_length.median(), session_length.mean(), len(data), data['SessionId'].nunique(), data['ItemId'].nunique()

(2, 191, 3.0, 4.133706087505183, 5254267, 1271079, 20163)

In [38]:
# cleanse_unpopular_item (3)
item_popular = data.groupby('ItemId').size() # ItemId의 도수분포 # of length 20163 # item은 20163 개 있다.
item_use = item_popular[item_popular >= least_click].index # of length 20159 # itemID의 도수가 5 이상인 것들의 index(ItemID)
data = data[data['ItemId'].isin(item_use)] # of length 5254251
# 5254251 = \sum_{item=1}^{20159}아이템별 클릭수
data_ui3 = data
session_length = data.groupby('SessionId').size()
session_length.min(), session_length.max(), session_length.median(), session_length.mean(), len(data), data['SessionId'].nunique(), data['ItemId'].nunique()

(1, 191, 3.0, 4.133700004012345, 5254251, 1271077, 20159)

In [39]:
after_len = len(data)
before_len - after_len

97

In [40]:
# cleanse_short_session (4)
before_len = len(data) # 5254251
session_len = data.groupby('SessionId').size() # sessionId의 도수분포 # of length 1271160 # sessionID는 1271077 개 있다.
session_use = session_len[session_len >= shortest].index # of length 1271074 # sessionID의 도수가 2 이상인 것들의 index(sessionID)
data = data[data['SessionId'].isin(session_use)] # of length 5254248   
# 5254248  = \sum_{세션=1}^{1271074}세션당 클릭수"
data_ss4 = data
session_length = data.groupby('SessionId').size()
session_length.min(), session_length.max(), session_length.median(), session_length.mean(), len(data), data['SessionId'].nunique(), data['ItemId'].nunique()

(2, 191, 3.0, 4.133707400198572, 5254248, 1271074, 20159)

In [41]:
# cleanse_unpopular_item (4)
item_popular = data.groupby('ItemId').size() # ItemId의 도수분포 # of length 20159 # item은 20159 개 있다.
item_use = item_popular[item_popular >= least_click].index # of length 20158 # itemID의 도수가 5 이상인 것들의 index(ItemID)
data = data[data['ItemId'].isin(item_use)] # of length 5254244 
# 5254244 = \sum_{item=1}^{20158}아이템별 클릭수
data_ui4 = data
session_length = data.groupby('SessionId').size()
session_length.min(), session_length.max(), session_length.median(), session_length.mean(), len(data), data['SessionId'].nunique(), data['ItemId'].nunique()

(1, 191, 3.0, 4.133704253253548, 5254244, 1271074, 20158)

In [42]:
after_len = len(data)
before_len - after_len

7

In [43]:
# cleanse_short_session (5)
before_len = len(data) # 5254244
session_len = data.groupby('SessionId').size() # sessionId의 도수분포 # of length 1271074 # sessionID는 1271074 개 있다.
session_use = session_len[session_len >= shortest].index # of length 1271072 # sessionID의 도수가 2 이상인 것들의 index(sessionID)
data = data[data['SessionId'].isin(session_use)] # of length 5254242   
# 5254242  = \sum_{세션=1}^{1271072}세션당 클릭수"
data_ss5 = data
session_length = data.groupby('SessionId').size()
session_length.min(), session_length.max(), session_length.median(), session_length.mean(), len(data), data['SessionId'].nunique(), data['ItemId'].nunique()

(2, 191, 3.0, 4.1337091840588105, 5254242, 1271072, 20158)

In [44]:
# cleanse_unpopular_item (5)
item_popular = data.groupby('ItemId').size() # ItemId의 도수분포 # of length 20158 # item은 20158 개 있다.
item_use = item_popular[item_popular >= least_click].index # of length 20158 # itemID의 도수가 5 이상인 것들의 index(ItemID)
data = data[data['ItemId'].isin(item_use)] # of length 5254242 
# 5254242 = \sum_{item=1}^{20158}아이템별 클릭수
data_ui5 = data
session_length = data.groupby('SessionId').size()
session_length.min(), session_length.max(), session_length.median(), session_length.mean(), len(data), data['SessionId'].nunique(), data['ItemId'].nunique()

(2, 191, 3.0, 4.1337091840588105, 5254242, 1271072, 20158)

In [45]:
after_len = len(data)
before_len - after_len

2

In [46]:
# cleanse_short_session (6)
before_len = len(data) # 5254242
session_len = data.groupby('SessionId').size() # sessionId의 도수분포 # of length 1271072 # sessionID는 1271072 개 있다.
session_use = session_len[session_len >= shortest].index # of length 1271072 # sessionID의 도수가 2 이상인 것들의 index(sessionID)
data = data[data['SessionId'].isin(session_use)] # of length 5254242   
# 5254242  = \sum_{세션=1}^{1271072}세션당 클릭수"
data_ss6 = data
session_length = data.groupby('SessionId').size()
session_length.min(), session_length.max(), session_length.median(), session_length.mean(), len(data), data['SessionId'].nunique(), data['ItemId'].nunique()

(2, 191, 3.0, 4.1337091840588105, 5254242, 1271072, 20158)

In [47]:
data = data_ss6
# cleanse_unpopular_item (6)
item_popular = data.groupby('ItemId').size() # ItemId의 도수분포 # of length 20158 # item은 20158 개 있다.
item_use = item_popular[item_popular >= least_click].index # of length 20158 # itemID의 도수가 5 이상인 것들의 index(ItemID)
data = data[data['ItemId'].isin(item_use)] # of length 5254242 
# 5254242 = \sum_{item=1}^{20158}아이템별 클릭수
data_ui6 = data
session_length = data.groupby('SessionId').size()
session_length.min(), session_length.max(), session_length.median(), session_length.mean(), len(data), data['SessionId'].nunique(), data['ItemId'].nunique()

(2, 191, 3.0, 4.1337091840588105, 5254242, 1271072, 20158)

In [48]:
after_len = len(data)
before_len - after_len
# end of the 'implementing manually'

0

In [83]:
# 우리가 사용할 데이터의 모양
data.shape

(5254242, 3)

In [87]:
data_after_2_4 = data

### 16-2-5. Train / Valid / Test split

In [76]:
test_path = data_path / 'yoochoose-test.dat'
test= load_data(test_path)
test['Time'].min(), test['Time'].max()

(Timestamp('2014-04-01 03:00:08.250000+0000', tz='UTC'),
 Timestamp('2014-09-30 02:59:23.866000+0000', tz='UTC'))

In [80]:
test_duplicated = test

In [78]:
test

,SessionId,Time,ItemId
0,5,2014-04-07 17:13:46.713000+00:00,214530776
1,5,2014-04-07 17:20:56.973000+00:00,214530776
2,5,2014-04-07 17:21:19.602000+00:00,214530776
3,10,2014-04-04 07:44:14.590000+00:00,214820942
4,10,2014-04-04 07:45:20.245000+00:00,214826810
...,...,...,...
8251786,11299820,2014-09-25 08:17:19.053000+00:00,214853094
8251787,11299815,2014-09-26 09:49:23.602000+00:00,214854804
8251788,11299815,2014-09-26 09:49:41.808000+00:00,214714715
8251789,11299810,2014-09-26 16:00:32.051000+00:00,214546123


In [134]:
# test를 불러와보니 data와 비슷하게 생겼다. 하지만 우리는 이 test를 사용하지 않을 것이다.
# data를 적당히 crop하여 train data, validation data, test data로 나눌 것이다.

In [168]:
def split_by_date(data: pd.DataFrame, n_days: int):
    final_time = data['Time'].max()
    session_last_time = data.groupby('SessionId')['Time'].max()
    session_in_train = session_last_time[session_last_time < final_time - dt.timedelta(n_days)].index
    session_in_test = session_last_time[session_last_time >= final_time - dt.timedelta(n_days)].index

    before_date = data[data['SessionId'].isin(session_in_train)]
    after_date = data[data['SessionId'].isin(session_in_test)]
    after_date = after_date[after_date['ItemId'].isin(before_date['ItemId'])]
    return before_date, after_date

In [182]:
data = data_after_2_4
tr, test = split_by_date(data, n_days=1)
tr, val = split_by_date(tr, n_days=1)

In [183]:
print(tr.shape)
print(test.shape)
print(val.shape)

(5125100, 3)
(71009, 3)
(58074, 3)


In [154]:
# split_by_date manually
# the first line : tr, test = split_by_date(data, n_days=1)
n_days=1
data = data_after_2_4
final_time = data['Time'].max() # data의 가장 마지막 시각은 2014-09-30 02:59:59이다.
session_last_time = data.groupby('SessionId')['Time'].max() # 각 SessionID에 대한 여러 Time 중 최댓값을 출력한다. (pd.Series)
session_in_train = session_last_time[session_last_time < final_time - dt.timedelta(n_days)].index 
session_in_test = session_last_time[session_last_time >= final_time - dt.timedelta(n_days)].index
# session_last_time의 날짜가 "final_time - 하루"보다 앞이면 train set으로, 뒤이면 test set으로 넣는 index를 생성한다.
# 즉, 마지막 거래일시가 final_time의 전날보다 앞에 있으면 train set 으로, final time의 전날보다 뒤에 있으면 test set으로 넣는다.

In [155]:
# 1271072 = 1255783 + 5289
print(len(session_last_time))
print(len(session_in_train))
print(len(session_in_test))
print(type(session_last_time))
print(type(session_in_train))
print(type(session_in_test))

1271072
1255783
15289
<class 'pandas.core.series.Series'>
<class 'pandas.core.indexes.numeric.Int64Index'>
<class 'pandas.core.indexes.numeric.Int64Index'>


In [156]:
before_date = data[data['SessionId'].isin(session_in_train)] # train set
after_date = data[data['SessionId'].isin(session_in_test)] 
after_date = after_date[after_date['ItemId'].isin(before_date['ItemId'])] # test set ; train set과 ItemId가 겹치는 경우만 고려한다.

In [157]:
# 위 코드의 세번째 행 때문에, 5183179 + 71009 =5254188 < 5254242 이다.
print(len(before_date))
print(len(after_date))
print(len(data))

5183179
71009
5254242


In [158]:
tr, test = before_date, after_date

In [159]:
# 잘 진행되고 있다. test.shape은 함수를 정의하여 했을 때와 일치하고 tr.shape은 다르다 (as desired)
print(tr.shape)
print(test.shape)

(5183179, 3)
(71009, 3)


In [160]:
# the second line : tr, val = split_by_date(tr, n_days=1)
n_days=1
final_time = tr['Time'].max()
session_last_time = tr.groupby('SessionId')['Time'].max()
session_in_train = session_last_time[session_last_time < final_time - dt.timedelta(n_days)].index 
session_in_test = session_last_time[session_last_time >= final_time - dt.timedelta(n_days)].index

In [161]:
before_tr = tr[data['SessionId'].isin(session_in_train)] # train set
after_tr = tr[data['SessionId'].isin(session_in_test)] 
after_tr = after_tr[after_tr['ItemId'].isin(before_tr['ItemId'])] # val set ; train set과 ItemId가 겹치는 경우만 고려한다.

In [162]:
tr, val = before_tr, after_tr

In [165]:
# 잘 진행되고 있다. 함수를 적용하여 진행했을 때와 일치한다.
print(tr.shape)
print(val.shape)
print(test.shape)

(5125100, 3)
(58074, 3)
(71009, 3)


In [167]:
tr.columns
# 어찌되었건, 위의 세 데이터셋이 우리가 사용할 데이터셋이다.
# 각각 512만 여개, 5만 여개, 7만 여개의 sample들로 이루어져 있다.
# columne들은 'SessionId', 'Time', 'ItemId' 이다.

Index(['SessionId', 'Time', 'ItemId'], dtype='object')

In [171]:
# data에 대한 정보를 살펴봅니다.
def stats_info(data: pd.DataFrame, status: str):
    print(f'* {status} Set Stats Info\n'
          f'\t Events: {len(data)}\n'
          f'\t Sessions: {data["SessionId"].nunique()}\n'
          f'\t Items: {data["ItemId"].nunique()}\n'
          f'\t First Time : {data["Time"].min()}\n'
          f'\t Last Time : {data["Time"].max()}\n')

In [172]:
# 이 코드들은 자세히 뜯어보지 않아도 알 수 있을 것 같다.
# Events가 바로 윗 셀에서 계산한 결과와 일치한다.
# 세션 수와 아이템 수는 계산하지는 않았지만, 계산하면 아래와 같다.
# 아이템 수 경우, "train 아이템수 > val 아이템수" 인 것은 당연하지만 더 나아가 "train 아이템 집합 \supset val 아이템 집합"이기도 하다.
# 마찬가지로 "train 아이템수 > test 아이템수" 이고 "train 아이템 집합 \supset test 아이템 집합"이다.
# Last TIme은 주목할 필요가 있다. test Last Time은 data의 마지막 시각과 일치한다.
# valid Last Time은 이것보다 약 하루 전이고, train Last Time은 거기서 또 하루 전이다.
# First Time에 대해서는 따로 처리를 해준 적이 없음에도 불구하고 train vaild test가 (거의) disjoint한 것처럼 보인다.
# 아마도 그건, 한 세션이 오래 지속되지 않았기 때문에 그런 것 같다.
stats_info(tr, 'train')
stats_info(val, 'valid')
stats_info(test, 'test')

* train Set Stats Info
	 Events: 5125100
	 Sessions: 1243431
	 Items: 20153
	 First Time : 2014-08-31 03:00:01.111000+00:00
	 Last Time : 2014-09-28 02:57:34.348000+00:00

* valid Set Stats Info
	 Events: 58074
	 Sessions: 12350
	 Items: 6232
	 First Time : 2014-09-28 03:00:25.298000+00:00
	 Last Time : 2014-09-29 02:58:27.660000+00:00

* test Set Stats Info
	 Events: 71009
	 Sessions: 15289
	 Items: 6580
	 First Time : 2014-09-29 02:37:20.695000+00:00
	 Last Time : 2014-09-30 02:59:59.430000+00:00



In [178]:
# 추측이 맞았다. 평균적으로 한 세션은 7분 51초동안 지속된다.
# 그러니 train valid test가 거의 disjoint하게 관측된다. vaild와 test는 disjoint하지 않지만, 그것 역시 plausible하다.
time_max = data.groupby('SessionId')['Time'].max()
time_min = data.groupby('SessionId')['Time'].min()
(time_max - time_min).mean()

Timedelta('0 days 00:07:51.132260743')

In [185]:
tr

,SessionId,Time,ItemId
26837834,9194111,2014-08-31 17:40:46.805000+00:00,214853420
26837835,9194111,2014-08-31 17:42:26.089000+00:00,214850942
26837836,9194111,2014-08-31 17:44:06.583000+00:00,214829878
26837837,9194111,2014-08-31 17:48:49.873000+00:00,214853420
26838202,9194123,2014-08-31 19:26:57.386000+00:00,214601207
...,...,...,...
32230485,11562157,2014-09-25 12:31:10.391000+00:00,214580372
32230486,11562157,2014-09-25 12:31:29.679000+00:00,214516012
32230487,11562158,2014-09-26 04:50:29.172000+00:00,214849132
32230488,11562158,2014-09-26 04:52:21.900000+00:00,214854774


In [186]:
# train set에 없는 아이템이 val, test기간에 생길 수 있으므로 train data를 기준으로 인덱싱합니다.
id2idx = {item_id : index for index, item_id in enumerate(tr['ItemId'].unique())}

def indexing(df, id2idx):
    df['item_idx'] = df['ItemId'].map(lambda x: id2idx.get(x, -1))  # id2idx에 없는 아이템은 모르는 값(-1) 처리 해줍니다.
    return df

tr = indexing(tr, id2idx)
val = indexing(val, id2idx)
test = indexing(test, id2idx)

In [190]:
# tr에 새로운 컬럼 item_idx가 생겼다.
tr

,SessionId,Time,ItemId,item_idx
26837834,9194111,2014-08-31 17:40:46.805000+00:00,214853420,0
26837835,9194111,2014-08-31 17:42:26.089000+00:00,214850942,1
26837836,9194111,2014-08-31 17:44:06.583000+00:00,214829878,2
26837837,9194111,2014-08-31 17:48:49.873000+00:00,214853420,0
26838202,9194123,2014-08-31 19:26:57.386000+00:00,214601207,3
...,...,...,...,...
32230485,11562157,2014-09-25 12:31:10.391000+00:00,214580372,9543
32230486,11562157,2014-09-25 12:31:29.679000+00:00,214516012,2798
32230487,11562158,2014-09-26 04:50:29.172000+00:00,214849132,1158
32230488,11562158,2014-09-26 04:52:21.900000+00:00,214854774,18665


In [192]:
# 인덱스(item_idx)가 잘 지정되었다. 코드에서 의도한 대로, tr에 먼저 인덱스를 부여하고, 그다음에 val, test 순으로 지정했다.
# 그렇기 때문에, tr의 인덱스들이 대체로 그 수가 적고, val과 test는 높다.
# val보다 test가 더 인덱스들이 클 것 같았는데, 그렇지는 않다. 하지만 그럴 수 있다.
print(tr['item_idx'].mean())
print(val['item_idx'].mean())
print(test['item_idx'].mean())

6092.2037187567075
12803.548627613045
12501.471545860384


In [180]:
# tr, val, test를 각각 피클 파일로 저장하는 것 같다.
save_path = data_path / 'processed'
save_path.mkdir(parents=True, exist_ok=True)

tr.to_pickle(save_path / 'train.pkl')
val.to_pickle(save_path / 'valid.pkl')
test.to_pickle(save_path / 'test.pkl')

In [59]:
save_path = data_path / 'processed'
save_path.mkdir(parents=True, exist_ok=True)

tr.to_pickle(save_path / 'train.pkl')
val.to_pickle(save_path / 'valid.pkl')
test.to_pickle(save_path / 'test.pkl')

## 16-3. 논문 소개

논문제목 : SESSION-BASED RECOMMENDATIONS WITH RECURRENT NEURAL NETWORKS

abstract만 간단히 읽어보자.

**We apply recurrent neural networks (RNN) on a new domain, namely recommender systems.**
Real-life recommender systems often face the problem of having to base recommendations only on **short session-based data** (e.g. a small sportsware website) instead of long user histories (as in the case of Netflix).
In this situation the frequently praised **matrix factorization approaches are not accurate**.
This problem is usually overcome in practice by resorting to item-to-item recommendations, i.e. recommending similar items.
We argue that by **modeling the whole session**, more accurate recommendations can be provided.
We therefore propose an **RNN-based approach for session-based recommendations**.
Our approach also considers practical aspects of the task and introduces several modifications to classic RNNs such as a **ranking loss function** that make it more viable(실행 가능한) for this specific problem.
Experimental results on two data-sets show marked improvements over widely used approaches.

## 16-4 Data Pipeline

In [60]:
class SessionDataset:
    """Credit to yhs-968/pyGRU4REC."""

    def __init__(self, data):
        self.df = data
        self.click_offsets = self.get_click_offsets()
        self.session_idx = np.arange(self.df['SessionId'].nunique())  # indexing to SessionId

    def get_click_offsets(self):
        """
        Return the indexes of the first click of each session IDs,
        """
        offsets = np.zeros(self.df['SessionId'].nunique() + 1, dtype=np.int32)
        offsets[1:] = self.df.groupby('SessionId').size().cumsum()
        return offsets

In [61]:
tr_dataset = SessionDataset(tr)
tr_dataset.df.head(10)

,SessionId,Time,ItemId,item_idx
26837834,9194111,2014-08-31 17:40:46.805000+00:00,214853420,0
26837835,9194111,2014-08-31 17:42:26.089000+00:00,214850942,1
26837836,9194111,2014-08-31 17:44:06.583000+00:00,214829878,2
26837837,9194111,2014-08-31 17:48:49.873000+00:00,214853420,0
26838202,9194123,2014-08-31 19:26:57.386000+00:00,214601207,3
26838203,9194123,2014-08-31 19:34:37.068000+00:00,214510689,4
26838193,9194124,2014-08-31 19:14:28.308000+00:00,214849327,5
26838194,9194124,2014-08-31 19:16:31.114000+00:00,214828970,6
26838196,9194127,2014-09-01 15:36:11.651000+00:00,214845997,7
26838197,9194127,2014-09-01 15:38:00.222000+00:00,214845997,7


In [62]:
tr_dataset.click_offsets

array([      0,       4,       6, ..., 5125095, 5125097, 5125100],
      dtype=int32)

In [63]:
tr_dataset.session_idx

array([      0,       1,       2, ..., 1243428, 1243429, 1243430])

In [64]:
class SessionDataLoader:
    """Credit to yhs-968/pyGRU4REC."""

    def __init__(self, dataset: SessionDataset, batch_size=50):
        self.dataset = dataset
        self.batch_size = batch_size

    def __iter__(self):
        """ Returns the iterator for producing session-parallel training mini-batches.
        Yields:
            input (B,):  Item indices that will be encoded as one-hot vectors later.
            target (B,): a Variable that stores the target item indices
            masks: Numpy array indicating the positions of the sessions to be terminated
        """

        start, end, mask, last_session, finished = self.initialize()  # initialize 메소드에서 확인해주세요.
        """
        start : Index Where Session Start
        end : Index Where Session End
        mask : indicator for the sessions to be terminated
        """

        while not finished:
            min_len = (end - start).min() - 1  # Shortest Length Among Sessions
            for i in range(min_len):
                # Build inputs & targets
                inp = self.dataset.df['item_idx'].values[start + i]
                target = self.dataset.df['item_idx'].values[start + i + 1]
                yield inp, target, mask

            start, end, mask, last_session, finished = self.update_status(start, end, min_len, last_session, finished)

    def initialize(self):
        first_iters = np.arange(self.batch_size)    # 첫 배치에 사용할 세션 Index를 가져옵니다.
        last_session = self.batch_size - 1    # 마지막으로 다루고 있는 세션 Index를 저장해둡니다.
        start = self.dataset.click_offsets[self.dataset.session_idx[first_iters]]       # data 상에서 session이 시작된 위치를 가져옵니다.
        end = self.dataset.click_offsets[self.dataset.session_idx[first_iters] + 1]  # session이 끝난 위치 바로 다음 위치를 가져옵니다.
        mask = np.array([])   # session의 모든 아이템을 다 돌은 경우 mask에 추가해줄 것입니다.
        finished = False         # data를 전부 돌았는지 기록하기 위한 변수입니다.
        return start, end, mask, last_session, finished

    def update_status(self, start: np.ndarray, end: np.ndarray, min_len: int, last_session: int, finished: bool):  
        # 다음 배치 데이터를 생성하기 위해 상태를 update합니다.
        
        start += min_len   # __iter__에서 min_len 만큼 for문을 돌았으므로 start를 min_len 만큼 더해줍니다.
        mask = np.arange(self.batch_size)[(end - start) == 1]  
        # end는 다음 세션이 시작되는 위치인데 start와 한 칸 차이난다는 것은 session이 끝났다는 뜻입니다. mask에 기록해줍니다.

        for i, idx in enumerate(mask, start=1):  # mask에 추가된 세션 개수만큼 새로운 세션을 돌것입니다.
            new_session = last_session + i  
            if new_session > self.dataset.session_idx[-1]:  # 만약 새로운 세션이 마지막 세션 index보다 크다면 모든 학습데이터를 돈 것입니다.
                finished = True
                break
            # update the next starting/ending point
            start[idx] = self.dataset.click_offsets[self.dataset.session_idx[new_session]]     # 종료된 세션 대신 새로운 세션의 시작점을 기록합니다.
            end[idx] = self.dataset.click_offsets[self.dataset.session_idx[new_session] + 1]

        last_session += len(mask)  # 마지막 세션의 위치를 기록해둡니다.
        return start, end, mask, last_session, finished

In [65]:
tr_data_loader = SessionDataLoader(tr_dataset, batch_size=4)
tr_dataset.df.head(15)

,SessionId,Time,ItemId,item_idx
26837834,9194111,2014-08-31 17:40:46.805000+00:00,214853420,0
26837835,9194111,2014-08-31 17:42:26.089000+00:00,214850942,1
26837836,9194111,2014-08-31 17:44:06.583000+00:00,214829878,2
26837837,9194111,2014-08-31 17:48:49.873000+00:00,214853420,0
26838202,9194123,2014-08-31 19:26:57.386000+00:00,214601207,3
26838203,9194123,2014-08-31 19:34:37.068000+00:00,214510689,4
26838193,9194124,2014-08-31 19:14:28.308000+00:00,214849327,5
26838194,9194124,2014-08-31 19:16:31.114000+00:00,214828970,6
26838196,9194127,2014-09-01 15:36:11.651000+00:00,214845997,7
26838197,9194127,2014-09-01 15:38:00.222000+00:00,214845997,7


In [66]:
iter_ex = iter(tr_data_loader)

In [67]:
inputs, labels, mask =  next(iter_ex)
print(f'Model Input Item Idx are : {inputs}')
print(f'Label Item Idx are : {"":5} {labels}')
print(f'Previous Masked Input Idx are {mask}')

Model Input Item Idx are : [0 3 5 7]
Label Item Idx are :       [1 4 6 7]
Previous Masked Input Idx are []


## 16-5 Modeling

In [68]:
def mrr_k(pred, truth: int, k: int):
    indexing = np.where(pred[:k] == truth)[0]
    if len(indexing) > 0:
        return 1 / (indexing[0] + 1)
    else:
        return 0


def recall_k(pred, truth: int, k: int) -> int:
    answer = truth in pred[:k]
    return int(answer)

In [69]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dropout, GRU
from tensorflow.keras.losses import categorical_crossentropy
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tqdm import tqdm

In [70]:
def create_model(args):
    inputs = Input(batch_shape=(args.batch_size, 1, args.num_items))
    gru, _ = GRU(args.hsz, stateful=True, return_state=True, name='GRU')(inputs)
    dropout = Dropout(args.drop_rate)(gru)
    predictions = Dense(args.num_items, activation='softmax')(dropout)
    model = Model(inputs=inputs, outputs=[predictions])
    model.compile(loss=categorical_crossentropy, optimizer=Adam(args.lr), metrics=['accuracy'])
    model.summary()
    return model

In [71]:
class Args:
    def __init__(self, tr, val, test, batch_size, hsz, drop_rate, lr, epochs, k):
        self.tr = tr
        self.val = val
        self.test = test
        self.num_items = tr['ItemId'].nunique()
        self.num_sessions = tr['SessionId'].nunique()
        self.batch_size = batch_size
        self.hsz = hsz
        self.drop_rate = drop_rate
        self.lr = lr
        self.epochs = epochs
        self.k = k

args = Args(tr, val, test, batch_size=2048, hsz=50, drop_rate=0.1, lr=0.001, epochs=3, k=20)

In [72]:
model = create_model(args)

Model: "model"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_1 (InputLayer)         [(2048, 1, 20153)]        0         
_________________________________________________________________
GRU (GRU)                    [(2048, 50), (2048, 50)]  3030750   
_________________________________________________________________
dropout (Dropout)            (2048, 50)                0         
_________________________________________________________________
dense (Dense)                (2048, 20153)             1027803   
Total params: 4,058,553
Trainable params: 4,058,553
Non-trainable params: 0
_________________________________________________________________


In [73]:
# train 셋으로 학습하면서 valid 셋으로 검증합니다.
def train_model(model, args):
    train_dataset = SessionDataset(args.tr)
    train_loader = SessionDataLoader(train_dataset, batch_size=args.batch_size)

    for epoch in range(1, args.epochs + 1):
        total_step = len(args.tr) - args.tr['SessionId'].nunique()
        tr_loader = tqdm(train_loader, total=total_step // args.batch_size, desc='Train', mininterval=1)
        for feat, target, mask in tr_loader:
            reset_hidden_states(model, mask)  # 종료된 session은 hidden_state를 초기화합니다. 아래 메서드에서 확인해주세요.

            input_ohe = to_categorical(feat, num_classes=args.num_items)
            input_ohe = np.expand_dims(input_ohe, axis=1)
            target_ohe = to_categorical(target, num_classes=args.num_items)

            result = model.train_on_batch(input_ohe, target_ohe)
            tr_loader.set_postfix(train_loss=result[0], accuracy = result[1])

        val_recall, val_mrr = get_metrics(args.val, model, args, args.k)  # valid set에 대해 검증합니다.

        print(f"\t - Recall@{args.k} epoch {epoch}: {val_recall:3f}")
        print(f"\t - MRR@{args.k}    epoch {epoch}: {val_mrr:3f}\n")


def reset_hidden_states(model, mask):
    gru_layer = model.get_layer(name='GRU')  # model에서 gru layer를 가져옵니다.
    hidden_states = gru_layer.states[0].numpy()  # gru_layer의 parameter를 가져옵니다.
    for elt in mask:  # mask된 인덱스 즉, 종료된 세션의 인덱스를 돌면서
        hidden_states[elt, :] = 0  # parameter를 초기화 합니다.
    gru_layer.reset_states(states=hidden_states)


def get_metrics(data, model, args, k: int):  # valid셋과 test셋을 평가하는 코드입니다. 
                                             # train과 거의 같지만 mrr, recall을 구하는 라인이 있습니다.
    dataset = SessionDataset(data)
    loader = SessionDataLoader(dataset, batch_size=args.batch_size)
    recall_list, mrr_list = [], []

    total_step = len(data) - data['SessionId'].nunique()
    for inputs, label, mask in tqdm(loader, total=total_step // args.batch_size, desc='Evaluation', mininterval=1):
        reset_hidden_states(model, mask)
        input_ohe = to_categorical(inputs, num_classes=args.num_items)
        input_ohe = np.expand_dims(input_ohe, axis=1)

        pred = model.predict(input_ohe, batch_size=args.batch_size)
        pred_arg = tf.argsort(pred, direction='DESCENDING')  # softmax 값이 큰 순서대로 sorting 합니다.

        length = len(inputs)
        recall_list.extend([recall_k(pred_arg[i], label[i], k) for i in range(length)])
        mrr_list.extend([mrr_k(pred_arg[i], label[i], k) for i in range(length)])

    recall, mrr = np.mean(recall_list), np.mean(mrr_list)
    return recall, mrr

In [74]:
# 학습 시간이 다소 오래 소요됩니다. 아래 주석을 풀지 마세요.
# train_model(model, args)

# 학습된 모델을 불러옵니다.
model = tf.keras.models.load_model(data_path / 'trained_model')

In [75]:
def test_model(model, args, test):
    test_recall, test_mrr = get_metrics(test, model, args, 20)
    print(f"\t - Recall@{args.k}: {test_recall:3f}")
    print(f"\t - MRR@{args.k}: {test_mrr:3f}\n")

test_model(model, args, test)

Evaluation:  81%|████████▏ | 22/27 [02:14<00:30,  6.12s/it]

	 - Recall@20: 0.711270
	 - MRR@20: 0.309212

